# 06. Construcción de una cartera candidata

## 1. ¿Qué le pedimos al optimizador?

En 05, el modelo exponencial quedó como referencia de trabajo a una sesión. Ahora usaremos esos escenarios para comparar dos asignaciones en una sola fecha de corte: pesos iguales y mínimo CVaR.

El mandato de esta primera versión es el siguiente:

- Ambas carteras usan el mismo universo elegible de 125 instrumentos en USD y los mismos escenarios exponenciales a una sesión, con semivida de 1,102 sesiones.
- Ambas están plenamente invertidas, sin posiciones cortas ni apalancamiento. Fijaremos el tope por instrumento antes de optimizar.
- Para cada escenario, el retorno de cartera es la suma de los retornos simples de los instrumentos multiplicados por sus pesos. La pérdida se expresa como el negativo de ese retorno.
- Usamos CVaR al 95% como criterio inicial: la pérdida media en el peor 5% de la masa de probabilidad. Con probabilidades desiguales, ese 5% no equivale necesariamente al 5% de las filas.
- Reportaremos el retorno medio estimado y la concentración para entender qué cambia al minimizar el CVaR. No se impondrá una meta de retorno en esta primera versión.

La notebook 06 termina con una tabla de pesos y una comparación de riesgo dentro de los escenarios. La evaluación histórica con rebalanceo y costos queda para 07.

**Siguiente pregunta:** ¿Cómo convertimos los retornos conjuntos de cada escenario en una distribución de pérdidas de cartera?

## 2. ¿Cuánto gana o pierde la cartera en cada escenario?

Los **pesos de cartera** asignan el capital entre instrumentos: un peso por cada una de las 125 columnas. Las **probabilidades de escenario** asignan masa a las fechas históricas: un peso por cada una de las 2,204 filas. Son objetos distintos: que una fecha reciente tenga más probabilidad no cambia directamente la asignación entre instrumentos, y que una cartera tenga un peso alto en un instrumento no hace más probable ningún escenario.

Para un escenario histórico $t$, el retorno simple de cartera es $r_{p,t}=r_t^\top w$, donde $r_t$ es el vector de retornos simples conjuntos y $w$ el vector de pesos de cartera. La pérdida se define como $L_t=-r_{p,t}$. En este bloque usamos probabilidades exponenciales sobre las filas históricas con semivida de 1,102 sesiones y una cartera plenamente invertida de pesos iguales; todavía no calculamos VaR, CVaR ni optimizamos.

In [1]:
from pathlib import Path
import site
import numpy as np

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        for root_candidate in (candidate, candidate / 'scenario-portfolio-lab', candidate / 'data-science-portfolio' / 'scenario-portfolio-lab'):
            if (root_candidate / 'config' / 'universe.csv').is_file() and (root_candidate / 'src' / 'scenario_portfolio_lab').is_dir():
                return root_candidate
    raise RuntimeError('Run this notebook inside the complete scenario-portfolio-lab repository.')

ROOT = find_project_root()
project_site_packages = ROOT / '.venv' / 'Lib' / 'site-packages'
if project_site_packages.is_dir():
    site.addsitedir(str(project_site_packages))
import polars as pl

CUT_RUN = ROOT / 'data' / 'published' / 'cut_2026-10-09_run_2026-10-11_full_window_final'
returns_path = CUT_RUN / 'gold' / 'eligible_simple_total_return_matrix.parquet'
eligible_path = CUT_RUN / 'reports' / 'eligible_universe.csv'
returns_with_initial_null = pl.read_parquet(returns_path)
eligible_catalog = pl.read_csv(eligible_path).filter(pl.col('universe_status') == 'ELIGIBLE')
# The first row has no approved preceding price. Remove that known initial-return row explicitly, not with a generic null filter.
initial_return_row = returns_with_initial_null.head(1)
if returns_with_initial_null.height != 2205 or initial_return_row.select(returns_with_initial_null.columns[1:]).null_count().to_numpy().sum() != 125:
    raise AssertionError('Expected exactly one initial all-null return row in the published matrix.')
returns_frame = returns_with_initial_null.slice(1, 2204)

scenario_dates = returns_frame['session_date'].to_list()
scenario_symbols = returns_frame.columns[1:]
simple_return_scenarios = returns_frame.select(scenario_symbols).to_numpy().astype(float)
SCENARIO_HALF_LIFE = 1102
scenario_count, instrument_count = simple_return_scenarios.shape
scenario_age = scenario_count - 1 - np.arange(scenario_count)
scenario_weights_exponential = np.exp(-np.log(2.0) * scenario_age / SCENARIO_HALF_LIFE)
scenario_weights_exponential /= scenario_weights_exponential.sum()
portfolio_weights_equal = np.full(instrument_count, 1.0 / instrument_count)

# Matrix multiplication: rows are historical joint scenarios; columns are instruments.
portfolio_returns_equal = simple_return_scenarios @ portfolio_weights_equal
portfolio_losses_equal = -portfolio_returns_equal

scenario_index_check = 0
explicit_contributions = simple_return_scenarios[scenario_index_check] * portfolio_weights_equal
block2_controls = {
    'published_cut_and_dimensions': scenario_count == 2204 and instrument_count == 125 and scenario_dates[-1].isoformat() == '2026-10-09',
    'dates_unique_and_increasing': scenario_dates == sorted(scenario_dates) and len(set(scenario_dates)) == scenario_count,
    'symbols_match_eligible_catalog_in_order': scenario_symbols == eligible_catalog['symbol'].to_list() and len(set(scenario_symbols)) == instrument_count,
    'initial_null_row_removed_and_remaining_returns_finite': returns_frame.null_count().to_numpy().sum() == 0 and np.isfinite(simple_return_scenarios).all() and np.all(simple_return_scenarios > -1.0),
    'scenario_probabilities_normalized': np.isfinite(scenario_weights_exponential).all() and np.all(scenario_weights_exponential > 0.0) and np.isclose(scenario_weights_exponential.sum(), 1.0, atol=1e-12, rtol=1e-12),
    'portfolio_weights_fully_invested': np.isfinite(portfolio_weights_equal).all() and np.all(portfolio_weights_equal >= 0.0) and np.isclose(portfolio_weights_equal.sum(), 1.0, atol=1e-12, rtol=1e-12),
    'scenario_return_matches_explicit_contributions': np.isclose(portfolio_returns_equal[scenario_index_check], explicit_contributions.sum(), atol=1e-12, rtol=1e-12),
    'loss_is_negative_portfolio_return': np.allclose(portfolio_losses_equal, -portfolio_returns_equal, atol=1e-12, rtol=1e-12),
}
if not all(block2_controls.values()):
    raise AssertionError([name for name, passed in block2_controls.items() if not passed])

scenario_preview_indices = [0, 1, 2, scenario_count - 2, scenario_count - 1]
scenario_preview = pl.DataFrame({
    'scenario_date': [scenario_dates[i] for i in scenario_preview_indices],
    'scenario_probability': scenario_weights_exponential[scenario_preview_indices],
    'equal_weight_portfolio_return': portfolio_returns_equal[scenario_preview_indices],
    'equal_weight_portfolio_loss': portfolio_losses_equal[scenario_preview_indices],
})
display(scenario_preview)
display(pl.DataFrame({'control': list(block2_controls), 'result': ['PASS' if passed else 'FAIL' for passed in block2_controls.values()]}))
print(f'Escenarios: {scenario_count}; instrumentos: {instrument_count}; semivida: {SCENARIO_HALF_LIFE} sesiones. Cada fecha mostrada identifica la procedencia histórica de un escenario conjunto.')


scenario_date,scenario_probability,equal_weight_portfolio_return,equal_weight_portfolio_loss
date,f64,f64,f64
2018-01-03,0.00021,0.004566,-0.004566
2018-01-04,0.00021,0.004631,-0.004631
2018-01-05,0.00021,0.006075,-0.006075
2026-10-08,0.000838,-0.001203,0.001203
2026-10-09,0.000838,0.007032,-0.007032


control,result
str,str
"""published_cut_and_dimensions""","""PASS"""
"""dates_unique_and_increasing""","""PASS"""
"""symbols_match_eligible_catalog…","""PASS"""
"""initial_null_row_removed_and_r…","""PASS"""
"""scenario_probabilities_normali…","""PASS"""
"""portfolio_weights_fully_invest…","""PASS"""
"""scenario_return_matches_explic…","""PASS"""
"""loss_is_negative_portfolio_ret…","""PASS"""


Escenarios: 2204; instrumentos: 125; semivida: 1102 sesiones. Cada fecha mostrada identifica la procedencia histórica de un escenario conjunto.
